In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


options = Options()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能


# 建立 Chrome 的 webdriver 物件
driver = webdriver.Chrome(options=options)

product = '麝香葡萄'


url = f"https://24h.pchome.com.tw/search/?q={product}"
# 開啟網頁
driver.get(url)

In [3]:
import time
all_result = []
page = 0

while True:
    time.sleep(5)
    elements = driver.find_elements(By.CSS_SELECTOR, 'ul.c-listInfoGrid__list.c-listInfoGrid__list--wrapProdCard.is-cardGap16X12 > li')
    print(len(elements))
    page += 1
    print(f'第{page}頁，有{len(elements)}筆資料')

    for element in elements:
        img_element = element.find_element(By.CSS_SELECTOR, 'div.c-prodInfoV2__img > img')
        img_url = img_element.get_attribute('src')
        title = img_element.get_attribute('alt')

        # 略過廣告
        price_elements = element.find_elements(By.CSS_SELECTOR, 'div.c-prodInfoV2__price')
        if price_elements == []:
            continue

        price = element.find_element(By.CSS_SELECTOR, 'div.c-prodInfoV2__price').text.replace('$', '').replace(',', '')
        link = element.find_element(By.CSS_SELECTOR, 'a.c-prodInfoV2__link.gtmClickV2').get_attribute('href')

        all_result.append({
            'img': img_url,
            'title': title,
            'price': price,
            'link': link
        })

    next_page_btn = driver.find_element(By.CSS_SELECTOR, 'div.c-pagination__button.is-next > button')

    # 有 disabled 這個 attribute 就會回傳 true (最後一頁), 沒有則None
    if next_page_btn.get_attribute('disabled'):
        print(f"\033[33m[End]\033[0m 已抵達PChome最後一頁 共 {len(all_result)} 筆")
        break
    
    next_page_btn.click()



40
第1頁，有40筆資料
5
第2頁，有5筆資料
[End] 已抵達PChome最後一頁 共 45 筆


In [4]:
# momo
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import json
from datetime import datetime

# 設定 Chrome 瀏覽器的選項
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") # Chrome 瀏覽器在啟動時最大化視窗
options.add_argument("--incognito") # 無痕模式
options.add_argument("--disable-popup-blocking") # 停用 Chrome 的彈窗阻擋功能。


def momo_products(product_name):
    driver = webdriver.Chrome(options=options)
    # product_name = "ROG滑鼠"
    driver.get(f"https://www.momoshop.com.tw/search/searchShop.jsp?keyword={product_name}&_isFuzzy=0&searchType=1")

    all_products = []

    while True:
        time.sleep(2)
        info_cards = driver.find_elements(By.CSS_SELECTOR, ".listAreaLi")
        print(f"有 {len(info_cards)} 個商品資訊")

        for card in info_cards:
            img_url = card.find_element(By.CSS_SELECTOR, "img.goods-img").get_attribute("src")
            link = card.find_element(By.CSS_SELECTOR, "a.prdName").get_attribute("href")
            title = card.find_element(By.CSS_SELECTOR, "a.prdName").text
            price = card.find_element(By.CSS_SELECTOR, "span.price > b").text

            all_products.append({
                "img_url": img_url,
                "link": link,
                "title": title,
                "price": price
            })

        next_buttons = driver.find_elements(By.CSS_SELECTOR, "div.page-btn.page-next > a")

        if next_buttons == []:
            print("已經是最後一頁，無法點擊下一頁按鈕")
            break
        
        driver.find_elements(By.CSS_SELECTOR, "div.page-btn.page-next > a")[1].click()
        
    driver.quit()

    with open(f"momo_{product_name}.json", "w", encoding="utf-8") as f:
        json.dump(all_products, f, ensure_ascii=False, indent=4)
